# March Machine Learning Mania 2026 - Baseline Model Training

Bu notebook'ta LightGBM kullanarak baseline model eğiteceğiz.

**Plan:**
1. Veri yükleme
2. Train/Val split (walk-forward)
3. LightGBM model eğitimi
4. Performans değerlendirme (Brier Score, AUC)
5. Feature importance analizi

## Cell 1: Imports ve Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Görüntüleme ayarları
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# Tüm sütunları göster
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Tüm kütüphaneler yüklendi!")
print(f"\nPython versiyonları:")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")
print(f"  lightgbm: {lgb.__version__}")

## Cell 2: Veri Yükleme

Feature engineering pipeline'ından çıkan processed CSV'leri yüklüyoruz.

In [ ]:
# Veri yükleme
men = pd.read_csv("../artifacts/data/processed_features_men.csv")
women = pd.read_csv("../artifacts/data/processed_features_women.csv")

print("=" * 60)
print("VERİ SETİ BİLGİLERİ")
print("=" * 60)
print(f"\nErkekler:")
print(f"  Toplam satır: {len(men):,}")
print(f"  Toplam sütun: {len(men.columns)}")
print(f"  Season aralığı: {men['Season'].min()} - {men['Season'].max()}")
print(f"  Target balance: {men['Target'].mean():.3f} (0.5 = dengeli)")

print(f"\nKadınlar:")
print(f"  Toplam satır: {len(women):,}")
print(f"  Toplam sütun: {len(women.columns)}")
print(f"  Season aralığı: {women['Season'].min()} - {women['Season'].max()}")
print(f"  Target balance: {women['Target'].mean():.3f} (0.5 = dengeli)")

# İlk birkaç satırı göster
print("\n" + "=" * 60)
print("ERKEKLER - İlk 5 satır:")
print("=" * 60)
display(men.head())

## Cell 3: Train/Val/Test Split (Walk-Forward)

**Neden Walk-Forward?**
- Time-series veri (yıl yıl ilerliyor)
- Geçmişle eğit, geleceği tahmin et
- Data leakage'ı önler

**Split Stratejisi:**
- Train: Season ≤ 2022 (geçmiş)
- Val: Season = 2023 (validation)
- Test: Season = 2024 (final test)

In [ ]:
def prepare_data(df, gender_label):
    """
    Veriyi train/val/test olarak böler.
    
    Returns:
        X_train, y_train, X_val, y_val, feature_columns
    """
    # Feature columns (_diff ile bitenler)
    feature_cols = [c for c in df.columns if c.endswith("_diff")]
    
    # Walk-forward split
    train = df[df["Season"] <= 2022]
    val = df[df["Season"] == 2023]
    test = df[df["Season"] == 2024]
    
    # X ve y ayrımı
    X_train = train[feature_cols]
    y_train = train["Target"]
    X_val = val[feature_cols]
    y_val = val["Target"]
    X_test = test[feature_cols]
    y_test = test["Target"]
    
    print(f"\n{gender_label} Data Split:")
    print(f"  Train (≤2022): {len(X_train):,} samples")
    print(f"  Val (2023):     {len(X_val):,} samples")
    print(f"  Test (2024):    {len(X_test):,} samples")
    print(f"  Features:       {len(feature_cols)}")
    
    # NaN kontrolü
    nan_count = X_train.isna().sum().sum()
    if nan_count > 0:
        print(f"  ⚠️  Uyarı: {nan_count} NaN değer var!")
    else:
        print(f"  ✅ NaN değer yok")
    
    return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols

# Erkekler için hazırla
X_train_m, y_train_m, X_val_m, y_val_m, X_test_m, y_test_m, features_m = prepare_data(men, "Erkekler")

# Kadınlar için hazırla
X_train_w, y_train_w, X_val_w, y_val_w, X_test_w, y_test_w, features_w = prepare_data(women, "Kadınlar")

## Cell 4: LightGBM Model Tanımı

**LightGBM Parametreleri Açıklaması:**

| Parametre | Değer | Açıklama |
|-----------|-------|----------|
| `objective` | 'binary' | Binary classification (kazanır/kazanmaz) |
| `metric` | 'binary_logloss' | Log loss minimizasyon |
| `n_estimators` | 500 | 500 karar ağacı |
| `learning_rate` | 0.05 | Her adımda ne kadar öğrenilsin |
| `max_depth` | 6 | Ağaç max derinliği (overfitting önler) |
| `num_leaves` | 31 | Yaprak sayısı |
| `min_child_samples` | 20 | Her yaprakta min sample (overfitting önler) |
| `verbose` | -1 | Çıktı kapat |

In [ ]:
def train_lgbm(X_train, y_train, X_val, y_val, gender_label):
    """
    LightGBM modelini eğitir.
    
    Returns:
        Trained model
    """
    print(f"\n{gender_label} modeli eğitiliyor...")
    
    # Model parametreleri (baseline)
    params = {
        'objective': 'binary',           # Binary classification
        'metric': 'binary_logloss',      # Log loss minimizasyon
        'n_estimators': 500,             # 500 ağaç
        'learning_rate': 0.05,           # Öğrenme hızı
        'max_depth': 6,                  # Ağaç derinliği
        'num_leaves': 31,                # Yaprak sayısı
        'min_child_samples': 20,         # Overfitting önleme
        'verbose': -1,                   # Çıktı kapat
        'random_state': 42               # Reproducibility
    }
    
    # Model oluştur
    model = lgb.LGBMClassifier(**params)
    
    # Eğit (early stopping ile)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    print(f"✅ {gender_label} modeli eğitildi!")
    print(f"   Kullanılan ağaç sayısı: {model.n_estimators_}")
    
    return model

## Cell 5: Model Eğitimi

Şimdi hem erkekler hem kadınlar için modelleri eğitiyoruz.

In [ ]:
# Erkekler modelini eğit
men_model = train_lgbm(X_train_m, y_train_m, X_val_m, y_val_m, "Erkekler")

# Kadınlar modelini eğit
women_model = train_lgbm(X_train_w, y_train_w, X_val_w, y_val_w, "Kadınlar")

print("\n" + "=" * 60)
print("TÜM MODELLER EĞİTİLDİ!")
print("=" * 60)

## Cell 6: Tahmin ve Performans Değerlendirme

**Metrikler Açıklaması:**

| Metrik | Ne ölçer? | İyi yön | Hedef |
|--------|----------|---------|-------|
| **Brier Score** | Olasılık kalibrasyonu | Düşük = iyi | < 0.20 |
| **Log Loss** | Probability error | Düşük = iyi | - |
| **AUC-ROC** | Separation gücü | Yüksek = iyi | > 0.75 |

**Brier Score:** `(predicted - actual)²` ortalaması
- Kaggle'ın kullandığı metric
- Overconfident tahminler ceza alır
- 0 = mükemmel, 0.25 = random guess, 1 = tam ters

In [ ]:
def evaluate_model(model, X_val, y_val, gender_label):
    """
    Model performansını değerlendirir.
    """
    # Tahminler (olasılık)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # Metrikler
    brier = brier_score_loss(y_val, y_pred_proba)
    logloss = log_loss(y_val, y_pred_proba)
    auc = roc_auc_score(y_val, y_pred_proba)
    
    # Sonuçları yazdır
    print(f"\n{'='*60}")
    print(f"{gender_label} MODEL PERFORMANSI (Validation Set)")
    print(f"{'='*60}")
    print(f"  Brier Score:  {brier:.4f}  {'✅' if brier < 0.20 else '⚠️'} (hedef: < 0.20)")
    print(f"  Log Loss:     {logloss:.4f}  (düşük = iyi)")
    print(f"  AUC-ROC:      {auc:.4f}  {'✅' if auc > 0.75 else '⚠️'} (hedef: > 0.75)")
    
    # Probability dağılımı
    print(f"\n  Probability Dağılımı:")
    print(f"    Min: {y_pred_proba.min():.3f}")
    print(f"    Mean: {y_pred_proba.mean():.3f}")
    print(f"    Max: {y_pred_proba.max():.3f}")
    
    # Gerçek vs Tahmin karşılaştırması
    actual_win_rate = y_val.mean()
    pred_win_rate = y_pred_proba.mean()
    print(f"\n  Gerçek kazanma oranı:  {actual_win_rate:.3f}")
    print(f"  Tahmin kazanma oranı:  {pred_win_rate:.3f}")
    print(f"  Fark: {abs(actual_win_rate - pred_win_rate):.3f} {'✅' if abs(actual_win_rate - pred_win_rate) < 0.05 else '⚠️'}")
    
    return y_pred_proba, brier, logloss, auc

# Erkekleri değerlendir
men_pred, men_brier, men_logloss, men_auc = evaluate_model(men_model, X_val_m, y_val_m, "ERKEKLER")

# Kadınları değerlendir
women_pred, women_brier, women_logloss, women_auc = evaluate_model(women_model, X_val_w, y_val_w, "KADINLAR")

print(f"\n{'='*60}")
print(f"ÖZET KARŞILAŞTIRMA")
print(f"{'='*60}")
print(f"{'Metric':<15} {'Erkekler':<15} {'Kadınlar':<15}")
print(f"{'-'*45}")
print(f"{'Brier Score':<15} {men_brier:<15.4f} {women_brier:<15.4f}")
print(f"{'AUC-ROC':<15} {men_auc:<15.4f} {women_auc:<15.4f}")

## Cell 7: Feature Importance Analizi

Hangi feature'lar model için en önemli? Bu analiz bize:
1. Hangi değişkenleri korumalıyız
2. Hangi değişkenleri çıkarabiliriz
3. Domain knowledge'ımız doğru mu

In [ ]:
def plot_feature_importance(model, features, gender_label, top_n=15):
    """
    Feature importance grafiği çizer.
    """
    # Importance verisi
    importance_df = pd.DataFrame({
        'Feature': features,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    # Top N
    top_features = importance_df.head(top_n)
    
    # Grafik
    plt.figure(figsize=(10, 8))
    plt.barh(top_features['Feature'], top_features['Importance'])
    plt.gca().invert_yaxis()
    plt.title(f'{gender_label} - Feature Importance (Top {top_n})', fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Tablo
    print(f"\n{gender_label} - Top {top_n} Feature:")
    print(top_features.to_string(index=False))
    
    # En zayıf 5 feature
    bottom_features = importance_df.tail(5)
    print(f"\n{gender_label} - En Zayıf 5 Feature (Drop adayı):")
    print(bottom_features.to_string(index=False))
    
    return importance_df

# Erkekler importance
men_importance = plot_feature_importance(men_model, features_m, "ERKEKLER", top_n=15)

# Kadınlar importance
women_importance = plot_feature_importance(women_model, features_w, "KADINLAR", top_n=15)

## Cell 8: Prediction Calibration Analysis

Modelimiz overconfident mi? Yani "%90 kazanır" dediğinde gerçekten %90 mı kazanıyor?

**İyi kalibre edilmiş model:** Tahmin ettiği olasılık ≈ Gerçekleşme oranı

In [ ]:
def analyze_calibration(y_true, y_pred_proba, gender_label, n_bins=5):
    """
    Model kalibrasyonunu analiz eder.
    """
    # Bins oluştur
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    # Her bin için actual win rate hesapla
    actual_rates = []
    pred_rates = []
    counts = []
    
    for i in range(n_bins):
        mask = (y_pred_proba > bins[i]) & (y_pred_proba <= bins[i + 1])
        if mask.sum() > 0:
            actual_rates.append(y_true[mask].mean())
            pred_rates.append(y_pred_proba[mask].mean())
            counts.append(mask.sum())
    
    # DataFrame
    calib_df = pd.DataFrame({
        'Pred_Prob': pred_rates,
        'Actual_Rate': actual_rates,
        'Count': counts
    })
    
    # Kalibrasyon error (ortalama fark)
    calib_error = np.mean(np.array(calib_df['Pred_Prob']) - np.array(calib_df['Actual_Rate']))
    
    print(f"\n{gender_label} - Kalibrasyon Analizi:")
    print(f"  Ortalama Kalibrasyon Error: {calib_error:.4f} {'✅' if abs(calib_error) < 0.05 else '⚠️'}")
    print(f"\n  Bin Analizi:")
    print(calib_df.round(3).to_string(index=False))
    
    # Grafik
    plt.figure(figsize=(8, 6))
    plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
    plt.scatter(pred_rates, actual_rates, s=counts, alpha=0.6)
    plt.xlabel('Predicted Probability', fontsize=12)
    plt.ylabel('Actual Win Rate', fontsize=12)
    plt.title(f'{gender_label} - Calibration Curve', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return calib_df

# Erkekler kalibrasyon
men_calib = analyze_calibration(y_val_m, men_pred, "ERKEKLER")

# Kadınlar kalibrasyon
women_calib = analyze_calibration(y_val_w, women_pred, "KADINLAR")

## Cell 9: Sonuç Özeti ve Model Kaydetme

In [ ]:
import joblib
from datetime import datetime

# Sonuç özeti
results = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'men': {
        'brier_score': men_brier,
        'log_loss': men_logloss,
        'auc': men_auc,
        'n_features': len(features_m),
        'n_estimators': men_model.n_estimators_
    },
    'women': {
        'brier_score': women_brier,
        'log_loss': women_logloss,
        'auc': women_auc,
        'n_features': len(features_w),
        'n_estimators': women_model.n_estimators_
    }
}

print("\n" + "="*60)
print("BASELINE MODEL EĞİTİMİ TAMAMLANDI!")
print("="*60)
print(f"\nZaman: {results['timestamp']}")
print(f"\nErkekler:")
print(f"  Brier Score: {men_brier:.4f}")
print(f"  AUC: {men_auc:.4f}")
print(f"  Kullanılan ağaç: {men_model.n_estimators_}")
print(f"\nKadınlar:")
print(f"  Brier Score: {women_brier:.4f}")
print(f"  AUC: {women_auc:.4f}")
print(f"  Kullanılan ağaç: {women_model.n_estimators_}")

# Modelleri kaydet
model_dir = '../artifacts/models'
import os
os.makedirs(model_dir, exist_ok=True)

joblib.dump(men_model, f'{model_dir}/men_baseline_model.pkl')
joblib.dump(women_model, f'{model_dir}/women_baseline_model.pkl')

print(f"\n✅ Modeller kaydedildi: {model_dir}/")

# Feature listesini kaydet
with open(f'{model_dir}/men_features.txt', 'w') as f:
    f.write('\n'.join(features_m))
with open(f'{model_dir}/women_features.txt', 'w') as f:
    f.write('\n'.join(features_w))

print(f"✅ Feature listeleri kaydedildi")

## 📝 Notlar ve Sonraki Adımlar

### Başarı Kriterleri:
- ✅ **Brier Score < 0.20**: İyi kalibre edilmiş
- ✅ **AUC > 0.75**: İyi ayırt edicilik

### Sonraki İyileştirme Adımları:

1. **Feature Selection**: Zayıf feature'ları çıkar
   - Importance < 10 olanları düşür
   - Yeniden eğit, performans değişimi?

2. **Hyperparameter Tuning**: Optuna ile en iyi parametreleri bul
   ```python
   # learning_rate: [0.01, 0.1]
   # max_depth: [4, 10]
   # num_leaves: [15, 63]
   ```

3. **Probability Calibration**: Isotonic regression
   ```python
   from sklearn.calibration import CalibratedClassifierCV
   calibrated = CalibratedClassifierCV(model, method='isotonic')
   ```

4. **Ensemble Modelleri**:
   - LightGBM + XGBoost + Logistic Regression
   - Weighted average

5. **2026 Tahminleri**: Kaggle submission hazırla

### Hedefler:
- Kaggle Brier Score: < 0.18 (Top %10)
- Private leaderboard'da başarılı ol